# LeetCode 3407 — Substring Matching Pattern bằng Python

Link đề: https://leetcode.com/problems/substring-matching-pattern/ · Độ khó: **Easy** · Chủ đề: String, String Matching (khớp chuỗi), Two Pointers/Greedy

Skeleton LeetCode:

```python
class Solution:
    def hasMatch(self, s: str, p: str) -> bool:
        ...
```

Ghi chú: đề và ví dụ được dịch từ đề gốc (Biweekly Contest 147); mọi lời giải trong notebook đã được chạy và kiểm chứng bằng code thật.

## 1. Mục tiêu học

Sau notebook này bạn sẽ:

1. Hiểu đề: mẫu `p` chứa **đúng một** ký tự `'*'`, `'*'` thay được bằng **mọi dãy ký tự độ dài từ 0 trở lên**; hỏi `p` có thể trở thành **một chuỗi con (substring)** của `s` hay không.
2. Nắm các công cụ Python: **`str.split('*')`** để tách mẫu thành **phần trước** và **phần sau** dấu `*`, **`str.find(sub, start)`** và **`str.rfind(sub)`** (tìm ký tự/chuỗi con, trả `-1` nếu không có), **`str.startswith(prefix, start)`**, chuỗi rỗng `""`, và (tùy chọn) **`re.search`**, **`fnmatch`**.
3. Đi từ cách **thử mọi chuỗi con của `s`** → **thử mọi vị trí bắt đầu của phần trước** → **tham lam: tìm phần trước sớm nhất rồi tìm phần sau từ chỗ đó** → **`find` đầu tiên + `rfind` cuối cùng** → **biểu thức chính quy `tiền_tố.*hậu_tố`**.
4. Hiểu mấu chốt: `p = A + '*' + B` khớp một chuỗi con ⇔ trong `s` có `A` xuất hiện **trước** (và **không chồng lên**) một lần xuất hiện của `B`; chọn `A` xuất hiện **sớm nhất** và `B` xuất hiện **muộn nhất** là đủ.
5. Tránh các lỗi kinh điển: chỉ kiểm `A in s and B in s` (bỏ qua **thứ tự** và **chồng lấp**), dùng `find` cho `B` thay vì tìm **từ sau `A`**, quên rằng `A` hoặc `B` có thể là **chuỗi rỗng** (`"u*"`, `"*c"`, `"*"`), và xử lý `'*'` như một ký tự thường.

## 2. Đề bài & ví dụ

Cho chuỗi `s` và chuỗi mẫu `p`, trong đó `p` chứa **đúng một** ký tự `'*'`. Ký tự `'*'` trong `p` có thể được thay bằng **bất kỳ dãy ký tự nào có độ dài `>= 0`** (kể cả dãy rỗng).

Trả về `true` nếu **sau khi thay `'*'`** ta có thể làm cho `p` trở thành **một chuỗi con** của `s`, ngược lại trả về `false`. Chuỗi con là một đoạn **liên tiếp** các ký tự của `s`.

**Ví dụ 1**

- Input: `s = "leetcode"`, `p = "ee*e"`
- Output: `true`
- Giải thích: thay `'*'` bằng `"tcod"` ta được `"eetcode"`, là chuỗi con của `s`.

**Ví dụ 2**

- Input: `s = "car"`, `p = "c*v"`
- Output: `false`
- Giải thích: không có chuỗi con nào khớp mẫu (`s` không có chữ `v`).

**Ví dụ 3**

- Input: `s = "luck"`, `p = "u*"`
- Output: `true`
- Giải thích: các chuỗi con `"u"`, `"uc"`, `"uck"` đều khớp.

**Ràng buộc:** `1 <= s.length <= 50`; `1 <= p.length <= 50`; `s` chỉ gồm chữ thường tiếng Anh; `p` gồm chữ thường tiếng Anh và **đúng một** `'*'`.

In [1]:
import random
import re
from fnmatch import fnmatchcase

examples = [
    ("leetcode", "ee*e", True),
    ("car", "c*v", False),
    ("luck", "u*", True),
]
for s, p, expected in examples:
    print(repr(s), repr(p), "-> kỳ vọng", expected)

'leetcode' 'ee*e' -> kỳ vọng True
'car' 'c*v' -> kỳ vọng False
'luck' 'u*' -> kỳ vọng True


## 3. Kiến thức nền

### 3.1. Tách mẫu bằng `split('*')`

`"ee*e".split("*")` cắt chuỗi tại ký tự `*` và trả về **list các phần**. Vì `p` có **đúng một** `*`, ta luôn nhận **đúng hai phần**: phần **trước** dấu `*` (gọi là `A`, tiền tố của mẫu) và phần **sau** (gọi là `B`, hậu tố của mẫu). Có thể một phần là **chuỗi rỗng** `""` nếu `*` nằm ở đầu hoặc cuối.

In [2]:
print("ee*e".split("*"))       # ['ee', 'e']
print("u*".split("*"))         # ['u', '']   (phần sau rỗng)
print("*c".split("*"))         # ['', 'c']   (phần trước rỗng)
print("*".split("*"))          # ['', '']    (cả hai rỗng)
A, B = "ee*e".split("*")       # giải nén hai phần
print("A =", repr(A), "| B =", repr(B))

['ee', 'e']
['u', '']
['', 'c']
['', '']
A = 'ee' | B = 'e'


### 3.2. `str.find(sub, start)` và `str.rfind(sub)`

- `s.find(sub)` trả về **chỉ số xuất hiện đầu tiên (từ trái)** của `sub` trong `s`, hoặc **`-1`** nếu không có.
- `s.find(sub, start)` chỉ tìm từ chỉ số `start` trở đi.
- `s.rfind(sub)` trả về chỉ số xuất hiện **cuối cùng (từ phải)**; cũng `-1` nếu không có.

`-1` **không phải chỉ số hợp lệ** để dùng — phải kiểm tra trước. (Đừng nhầm với `s.index`, hàm này **ném lỗi** `ValueError` khi không có.)

In [3]:
s = "leetcode"
print(s.find("e"))             # 1   (e đầu tiên)
print(s.find("e", 3))          # 7   (e đầu tiên kể từ chỉ số 3)
print(s.rfind("e"))            # 7   (e cuối cùng)
print(s.find("z"))             # -1  (không có)
print(s.find("ee"), s.rfind("ee"))   # 1 1

1
7
7
-1
1 1


### 3.3. Chuỗi rỗng `""` và hành vi đặc biệt của `find`

Chuỗi rỗng là chuỗi con của **mọi** chuỗi. `s.find("")` là `0`; `s.find("", k)` là `k` nếu `k <= len(s)`; `s.rfind("")` là `len(s)`. Nhờ vậy khi `A` hoặc `B` rỗng, các phép `find` vẫn cho kết quả hợp lý mà **không cần viết nhánh riêng**.

In [4]:
s = "luck"
print(s.find(""))              # 0
print(s.find("", 2))           # 2
print(s.find("", 4))           # 4   (start == len(s) vẫn hợp lệ)
print(s.rfind(""))             # 4   (= len(s))
print(s.find("", 5))           # -1  (start > len(s))

0
2
4
4
-1


### 3.4. `startswith(prefix, start)` và cắt chuỗi con

`s.startswith(sub, i)` cho biết `sub` có xuất hiện **bắt đầu tại chỉ số `i`** hay không — rất tiện khi ta thử từng vị trí. Cắt `s[i:j]` lấy chuỗi con từ `i` đến `j - 1`; số chuỗi con (kể cả rỗng) là `n(n + 1)/2 + 1`.

In [5]:
s = "leetcode"
print(s.startswith("ee", 1))          # True   (ee bắt đầu tại 1)
print(s.startswith("ee", 2))          # False
print(s.startswith("", 5))            # True   (rỗng luôn khớp)
print(s[1:4], "|", s[1:1] == "")      # eet | True

True
False
True
eet | True


### 3.5. (Tùy chọn) Biểu thức chính quy và `fnmatch`

Trong biểu thức chính quy (`re`) ký hiệu **`.*`** nghĩa là "bất kỳ chuỗi nào" — chính là vai trò của `'*'` trong đề. `re.search(mẫu, s)` tìm mẫu ở **bất kỳ vị trí nào** của `s` (đúng tinh thần "chuỗi con"). Module `fnmatch` có `fnmatchcase(text, mẫu)` hiểu `*` là "bất kỳ chuỗi nào" và khớp **toàn bộ** `text` — ta sẽ dùng nó làm **đáp án chuẩn** ở phần test.

In [6]:
print(bool(re.search("ee.*e", "leetcode")))       # True
print(bool(re.search("c.*v", "car")))             # False
print(fnmatchcase("eetcode", "ee*e"))             # True  (khớp toàn bộ "eetcode")
print(fnmatchcase("eet", "ee*e"))                 # False (phải kết thúc bằng e)

True
False
True
False


## 4. Ý tưởng nền tảng / cách nghĩ trực giác (tính tay)

Tách `p = A + '*' + B`. Một chuỗi con `t` của `s` khớp `p` khi và chỉ khi `t` **bắt đầu bằng `A`**, **kết thúc bằng `B`**, và đủ dài để `A` và `B` **không chồng nhau**: `len(t) >= len(A) + len(B)`. Phần `*` chính là "đoạn ở giữa" (có thể rỗng).

Với `s = "leetcode"`, `p = "ee*e"` (`A = "ee"`, `B = "e"`):

| chuỗi con `t` | bắt đầu bằng `ee`? | kết thúc bằng `e`? | `len >= 3`? | khớp? |
|---|---|---|---|---|
| `"ee"` | có | không | có | không |
| `"eet"` | có | không | có | không |
| `"eetcode"` | có | **có** | có | **có** |
| `"eetcod"` | có | không | có | không |

Chỉ cần **một** chuỗi con khớp là đủ ⇒ trả về `true`. Với `s = "car"`, `p = "c*v"`: không chuỗi con nào kết thúc bằng `v` ⇒ `false`.

In [7]:
def matches(t, A, B):
    """Chuỗi t có khớp mẫu A*B không?"""
    return t.startswith(A) and t.endswith(B) and len(t) >= len(A) + len(B)


s, A, B = "leetcode", "ee", "e"
for t in ["ee", "eet", "eetcode", "eetcod"]:
    print(repr(t), matches(t, A, B))

'ee' False
'eet' False
'eetcode' True
'eetcod' False


## 5. Cách 1 — Brute Force / baseline: thử mọi chuỗi con của `s`

**Ý tưởng.** Duyệt mọi cặp `(i, j)` với `0 <= i <= j <= n`, lấy `t = s[i:j]` và kiểm tra `matches(t, A, B)`. Cách này bám sát định nghĩa nhất.

In [8]:
def has_match_brute(s: str, p: str) -> bool:
    A, B = p.split("*")
    n = len(s)
    for i in range(n + 1):
        for j in range(i, n + 1):
            t = s[i:j]                                    # chuỗi con s[i..j-1]
            if t.startswith(A) and t.endswith(B) and len(t) >= len(A) + len(B):
                return True
    return False


for s, p, expected in examples:
    print(has_match_brute(s, p), "(kỳ vọng", expected, ")")

True (kỳ vọng True )
False (kỳ vọng False )
True (kỳ vọng True )


**Độ phức tạp (Cách 1).** Gọi `n = len(s)`, `m = len(p)`. Có `O(n²)` chuỗi con; mỗi chuỗi cắt và so sánh tốn `O(n + m)` ⇒ **Time `O(n² · (n + m))`** (tức `O(n³)` khi `n ~ m`), **Space `O(n)`** cho chuỗi con. Với `n, m <= 50` vẫn chạy ngay.

**Nút thắt:** ta thử cả những chuỗi con **không bắt đầu bằng `A`** rồi mới loại. Ta nên chỉ xét các vị trí mà `A` **thật sự xuất hiện**, rồi tìm `B` **ở phía sau**.

## 6. Ý tưởng thuật toán / bước đột phá

**Khớp từng phần theo thứ tự.** `p = A * B` khớp ⇔ tồn tại hai vị trí `i` và `j` sao cho:

- `A` xuất hiện tại `s[i : i + len(A)]`,
- `B` xuất hiện tại `s[j : j + len(B)]`,
- và `i + len(A) <= j` (`B` bắt đầu **sau khi `A` kết thúc**, không chồng lên).

Phần ở giữa `s[i + len(A) : j]` chính là đoạn thay cho `*` (rỗng nếu bằng nhau).

**Tham lam.** Để có nhiều chỗ nhất cho `B`, ta chọn `A` **kết thúc sớm nhất** — tức `A` xuất hiện **đầu tiên** (từ trái). Rồi ta chỉ cần xem `B` có xuất hiện **từ vị trí đó trở đi** không: `s.find(B, i + len(A))`. Nếu có ⇒ `true`.

Đảo lại: ta cũng có thể chọn `B` **xuất hiện cuối cùng** (`rfind`) rồi so `i + len(A) <= j`. Hai cách tương đương.

In [9]:
s, A, B = "leetcode", "ee", "e"
i = s.find(A)                         # A xuất hiện đầu tiên
print("i =", i, "-> A kết thúc tại", i + len(A))
j = s.find(B, i + len(A))             # tìm B từ chỗ A kết thúc
print("j =", j, "-> B bắt đầu tại", j)
print("khớp:", i != -1 and j != -1, "| phần thay cho *:", repr(s[i + len(A):j]))

i = 1 -> A kết thúc tại 3
j = 7 -> B bắt đầu tại 7
khớp: True | phần thay cho *: 'tcod'


## 7. Các cách tiếp theo

### Cách 2 — Thử mọi vị trí bắt đầu của `A`, rồi tìm vị trí của `B`

**Giải thích từng đoạn.**

1. `for i in range(n + 1)`: thử mọi vị trí `i` làm điểm bắt đầu của `A`.
2. `if s.startswith(A, i)`: `A` có xuất hiện tại `i` không.
3. Với mỗi `i` hợp lệ, thử mọi `j` từ `i + len(A)` đến `n`: `s.startswith(B, j)` ⇒ khớp.

In [10]:
def has_match_positions(s: str, p: str) -> bool:
    A, B = p.split("*")
    n = len(s)
    for i in range(n + 1):
        if not s.startswith(A, i):                # A không bắt đầu tại i
            continue
        for j in range(i + len(A), n + 1):        # B phải bắt đầu sau khi A kết thúc
            if s.startswith(B, j):
                return True
    return False


for s, p, expected in examples:
    print(has_match_positions(s, p), "(kỳ vọng", expected, ")")

True (kỳ vọng True )
False (kỳ vọng False )
True (kỳ vọng True )


In [11]:
# Code nộp LeetCode cho Cách 2 (đổi tên lớp thành Solution khi nộp)
class SolutionPositions:
    def hasMatch(self, s: str, p: str) -> bool:
        A, B = p.split("*")
        n = len(s)
        for i in range(n + 1):
            if not s.startswith(A, i):
                continue
            for j in range(i + len(A), n + 1):
                if s.startswith(B, j):
                    return True
        return False


print(SolutionPositions().hasMatch("leetcode", "ee*e"))     # True

True


**Độ phức tạp Cách 2:** `O(n)` giá trị `i`, mỗi `i` có `O(n)` giá trị `j`, mỗi lần `startswith` tốn `O(m)` ⇒ **Time `O(n² · m)`**, **Space `O(1)`** phụ (không kể `A`, `B`).

### Cách 3 — Tham lam: `find` phần đầu, rồi `find` phần sau từ chỗ đó

**Giải thích.** Dùng đúng ý tưởng mục 6: tìm `A` sớm nhất, rồi tìm `B` bắt đầu từ vị trí `A` kết thúc. Không cần hai vòng `for`.

In [12]:
def has_match_greedy(s: str, p: str) -> bool:
    A, B = p.split("*")
    i = s.find(A)                                  # A sớm nhất
    if i == -1:
        return False
    j = s.find(B, i + len(A))                      # B từ chỗ A kết thúc trở đi
    return j != -1


for s, p, expected in examples:
    print(has_match_greedy(s, p), "(kỳ vọng", expected, ")")

True (kỳ vọng True )
False (kỳ vọng False )
True (kỳ vọng True )


In [13]:
# Code nộp LeetCode cho Cách 3
class SolutionGreedy:
    def hasMatch(self, s: str, p: str) -> bool:
        A, B = p.split("*")
        i = s.find(A)
        if i == -1:
            return False
        return s.find(B, i + len(A)) != -1


print(SolutionGreedy().hasMatch("luck", "u*"))              # True

True


**Độ phức tạp Cách 3:** hai lần `find`, mỗi lần tối đa `O(n · m)` (cài đặt đơn giản) ⇒ **Time `O(n · m)`**, **Space `O(1)`** (không kể hai phần `A`, `B`).

### Cách 4 — `find` đầu tiên cho `A`, `rfind` cuối cùng cho `B`

**Giải thích.** `i = s.find(A)` (sớm nhất), `j = s.rfind(B)` (muộn nhất). `A` và `B` không chồng nhau khi `i + len(A) <= j`. Ngắn gọn và đối xứng: "đầu mẫu" càng sớm càng tốt, "cuối mẫu" càng muộn càng tốt.

In [14]:
def has_match_find_rfind(s: str, p: str) -> bool:
    A, B = p.split("*")
    i = s.find(A)                  # lần xuất hiện sớm nhất của A
    j = s.rfind(B)                 # lần xuất hiện muộn nhất của B
    return i != -1 and j != -1 and i + len(A) <= j


for s, p, expected in examples:
    print(has_match_find_rfind(s, p), "(kỳ vọng", expected, ")")

True (kỳ vọng True )
False (kỳ vọng False )
True (kỳ vọng True )


In [15]:
# Code nộp LeetCode cho Cách 4
class SolutionFindRfind:
    def hasMatch(self, s: str, p: str) -> bool:
        A, B = p.split("*")
        i, j = s.find(A), s.rfind(B)
        return i != -1 and j != -1 and i + len(A) <= j


print(SolutionFindRfind().hasMatch("car", "c*v"))           # False

False


**Độ phức tạp Cách 4:** **Time `O(n · m)`**, **Space `O(1)`**. Các trường hợp rỗng đúng nhờ mục 3.3: `A = ""` ⇒ `i = 0`; `B = ""` ⇒ `j = len(s)`.

### Cách 5 — Biểu thức chính quy `A.*B`

**Giải thích.** Thay `*` bằng `.*` rồi tìm trong `s`: `re.search(re.escape(A) + ".*" + re.escape(B), s)`. `re.escape` bảo vệ các ký tự đặc biệt (với chữ thường không cần, nhưng là thói quen tốt). `search` tìm ở mọi vị trí nên tương ứng với "chuỗi con". Cách này **ngắn nhưng che giấu thuật toán**, nên chỉ dùng để đối chiếu (hoặc khi thật sự cần mẫu phức tạp).

In [16]:
def has_match_regex(s: str, p: str) -> bool:
    A, B = p.split("*")
    pattern = re.escape(A) + ".*" + re.escape(B)
    return re.search(pattern, s) is not None


for s, p, expected in examples:
    print(has_match_regex(s, p), "(kỳ vọng", expected, ")")

True (kỳ vọng True )
False (kỳ vọng False )
True (kỳ vọng True )


**Độ phức tạp Cách 5:** với mẫu dạng này engine `re` dò từng vị trí và quay lui ở `.*` ⇒ trường hợp xấu `O(n²)` hoặc hơn; với `n <= 50` không đáng kể. **Space `O(m)`** cho mẫu đã biên dịch.

## 8. TRACE từng bước

Ta trace Cách 3 trên hai ví dụ: `("leetcode", "ee*e")` và ví dụ **có bẫy** `("bab", "a*b")` — nơi dùng `find` thường (thay vì tìm từ vị trí sau `A`) cho `B` sẽ sai.

| bước | `s` | `A`, `B` | `i = find(A)` | `A` kết thúc tại | `j = find(B, ·)` | kết luận |
|---|---|---|---|---|---|---|
| 1 | `leetcode` | `ee`, `e` | 1 | 3 | 7 | `j != -1` ⇒ **true** |
| 2 | `bab` | `a`, `b` | 1 | 2 | 2 | `j != -1` ⇒ **true** |

Ở dòng 2: `b` đầu tiên (chỉ số `0`) nằm **trước** `a` nên không dùng được; ta phải tìm `b` **từ chỉ số `2`**, và tìm thấy `b` ở chỉ số `2`. Chuỗi con khớp là `"ab"`.

In [17]:
def trace_has_match(s, p):
    A, B = p.split("*")
    print(f"s={s!r}, p={p!r} -> A={A!r}, B={B!r}")
    i = s.find(A)
    print(f"  i = s.find({A!r}) = {i}")
    if i == -1:
        print("  A không xuất hiện -> False")
        return False
    start = i + len(A)
    print(f"  A kết thúc tại {start}; tìm B từ chỉ số {start}")
    j = s.find(B, start)
    print(f"  j = s.find({B!r}, {start}) = {j}")
    if j == -1:
        print("  không tìm thấy B phía sau A -> False")
        return False
    print(f"  khớp: s[{i}:{j + len(B)}] = {s[i:j + len(B)]!r}, phần thay cho '*' = {s[start:j]!r} -> True")
    return True


trace_has_match("leetcode", "ee*e")
trace_has_match("bab", "a*b")
trace_has_match("car", "c*v")

s='leetcode', p='ee*e' -> A='ee', B='e'
  i = s.find('ee') = 1
  A kết thúc tại 3; tìm B từ chỉ số 3
  j = s.find('e', 3) = 7
  khớp: s[1:8] = 'eetcode', phần thay cho '*' = 'tcod' -> True
s='bab', p='a*b' -> A='a', B='b'
  i = s.find('a') = 1
  A kết thúc tại 2; tìm B từ chỉ số 2
  j = s.find('b', 2) = 2
  khớp: s[1:3] = 'ab', phần thay cho '*' = '' -> True
s='car', p='c*v' -> A='c', B='v'
  i = s.find('c') = 0
  A kết thúc tại 1; tìm B từ chỉ số 1
  j = s.find('v', 1) = -1
  không tìm thấy B phía sau A -> False


False

## 9. Code nộp LeetCode

Ta chọn **Cách 3** (tham lam: `find` `A`, rồi `find` `B` từ chỗ `A` kết thúc) hoặc **Cách 4**. Cả hai đều đúng; Cách 3 trực quan hơn vì thể hiện đúng thứ tự "trước rồi sau".

In [18]:
class Solution:
    def hasMatch(self, s: str, p: str) -> bool:
        A, B = p.split("*")                 # phần trước và phần sau dấu *
        i = s.find(A)                       # A xuất hiện sớm nhất (A rỗng -> 0)
        if i == -1:
            return False
        # B phải bắt đầu sau khi A kết thúc (không chồng lên A)
        return s.find(B, i + len(A)) != -1


for s, p, expected in examples:
    print(Solution().hasMatch(s, p), "(kỳ vọng", expected, ")")

True (kỳ vọng True )
False (kỳ vọng False )
True (kỳ vọng True )


## 10. Vì sao đúng

- **Định nghĩa tương đương.** `p = A * B` có thể trở thành chuỗi con của `s` ⇔ tồn tại `i, j` với `s[i : i + |A|] = A`, `s[j : j + |B|] = B` và `i + |A| <= j`. Khi đó chuỗi con `s[i : j + |B|]` bằng `A + (đoạn giữa) + B`, đoạn giữa `s[i + |A| : j]` thay cho `*` (độ dài `>= 0`).
- **Chọn `A` sớm nhất là đủ.** Giả sử có một cặp hợp lệ `(i', j')`. Gọi `i0 = s.find(A) <= i'` (lần xuất hiện sớm nhất). Khi đó `i0 + |A| <= i' + |A| <= j'`, nên `j'` vẫn nằm sau chỗ `A` kết thúc: `s.find(B, i0 + |A|)` **tìm thấy** một vị trí (nhiều nhất là `j'`). Vậy nếu có đáp án thì thuật toán trả `true`.
- **Ngược lại.** Nếu thuật toán trả `true`, cặp `(i0, j)` với `j >= i0 + |A|` chính là một cặp hợp lệ ⇒ có chuỗi con khớp. Vậy không có kết quả sai dương.
- **`A` hoặc `B` rỗng.** `s.find("") = 0` và `s.find("", k) = k` khi `k <= |s|`, nên các trường hợp `"*c"`, `"u*"`, `"*"` đều được xử lý đúng mà không cần nhánh riêng; `i + |A| <= |s|` luôn đúng khi `A` được tìm thấy nên `find(B, i + |A|)` không bị "start vượt biên".

## 11. Time/Space Complexity

Gọi `n = len(s)`, `m = len(p)`.

- **Time `O(n · m)`** trong trường hợp xấu với cách `find` đơn giản (mỗi lần `find` có thể so từng vị trí với tối đa `m` ký tự). Python dùng thuật toán tìm kiếm tối ưu nên thực tế nhanh hơn. Với `n, m <= 50` hằng số không quan trọng.
- **Space `O(m)`** cho hai phần `A`, `B` tạo bởi `split` (hoặc `O(1)` nếu coi như chỉ số).

Bài này có thể giải trong `O(n + m)` bằng **KMP/Z-function** nhưng với ràng buộc nhỏ thì `find` có sẵn là quá đủ.

## 12. Bộ test

### 12.1. Ví dụ của đề và gom các cách vào một bảng

In [19]:
impls = {
    "brute": has_match_brute,
    "positions": has_match_positions,
    "greedy": has_match_greedy,
    "find_rfind": has_match_find_rfind,
    "regex": has_match_regex,
    "Solution": lambda s, p: Solution().hasMatch(s, p),
}

for s, p, expected in examples:
    for name, fn in impls.items():
        assert fn(s, p) == expected, (name, s, p)
print("PASS các ví dụ của đề với", len(impls), "cách")

PASS các ví dụ của đề với 6 cách


### 12.2. Edge case

- `p = "*"`: khớp **mọi** `s` (thay bằng chuỗi rỗng).
- `A` rỗng (`"*c"`), `B` rỗng (`"u*"`).
- `A` và `B` **chồng nhau** trong `s` (không được dùng chung ký tự): `s = "abc"`, `p = "abc*bc"`.
- `B` xuất hiện **trước** `A` nhưng không sau: `s = "ba"`, `p = "a*b"`.
- Mẫu dài bằng đúng `s` (không có `*` ở giữa phần thay): `s = "abc"`, `p = "ab*c"`.
- `s` một ký tự: `"a"`/`"a*"`, `"a"`/`"*a"`, `"a"`/`"a*a"`.

In [20]:
cases = [
    ("a", "*", True),
    ("abc", "*", True),
    ("abc", "*c", True),
    ("abc", "a*", True),
    ("abc", "*d", False),
    ("abc", "d*", False),
    ("abc", "abc*bc", False),        # A và B chồng nhau: không hợp lệ
    ("abcbc", "abc*bc", True),       # đủ chỗ cho cả hai
    ("ba", "a*b", False),            # B chỉ có trước A
    ("bab", "a*b", True),            # B có thể tìm thấy sau A
    ("abc", "ab*c", True),
    ("abc", "abc*", True),
    ("abc", "*abc", True),
    ("a", "a*", True),
    ("a", "*a", True),
    ("a", "a*a", False),             # cần hai chữ a
    ("aa", "a*a", True),
    ("aaa", "aa*aa", False),
    ("aaaa", "aa*aa", True),
    ("leetcode", "l*e", True),
    ("leetcode", "e*l", False),      # sau mọi chữ e không còn chữ l... (l chỉ ở đầu)
    ("leetcode", "code*", True),
    ("leetcode", "*leet", True),
    ("leetcode", "d*d", False),
]
for s, p, expected in cases:
    for name, fn in impls.items():
        assert fn(s, p) == expected, (name, s, p, fn(s, p), expected)
print("PASS", len(cases), "edge case x", len(impls), "cách")

PASS 24 edge case x 6 cách


### 12.3. Phản ví dụ cho các cách làm sai

Ta dựng các hàm sai (mục 13) và chứng minh chúng cho kết quả khác đáp án đúng trên các đầu vào cụ thể.

In [21]:
def wrong_in_both(s, p):
    A, B = p.split("*")
    return A in s and B in s                      # SAI: bỏ qua thứ tự và chồng lấp


def wrong_plain_find(s, p):
    A, B = p.split("*")
    i, j = s.find(A), s.find(B)                   # SAI: B tìm từ đầu, không từ sau A
    return i != -1 and j != -1 and i + len(A) <= j


for s, p in [("ba", "a*b"), ("abc", "abc*bc"), ("bab", "a*b")]:
    print(f"s={s!r}, p={p!r} | đúng={Solution().hasMatch(s, p)} | wrong_in_both={wrong_in_both(s, p)} | wrong_plain_find={wrong_plain_find(s, p)}")

assert wrong_in_both("ba", "a*b") != Solution().hasMatch("ba", "a*b")
assert wrong_in_both("abc", "abc*bc") != Solution().hasMatch("abc", "abc*bc")
assert wrong_plain_find("bab", "a*b") != Solution().hasMatch("bab", "a*b")

s='ba', p='a*b' | đúng=False | wrong_in_both=True | wrong_plain_find=False
s='abc', p='abc*bc' | đúng=False | wrong_in_both=True | wrong_plain_find=False
s='bab', p='a*b' | đúng=True | wrong_in_both=True | wrong_plain_find=False


### 12.4. Đối chiếu ngẫu nhiên (có seed) với đáp án chuẩn độc lập

Đáp án chuẩn: duyệt **mọi chuỗi con** `t` của `s` (kể cả rỗng) và dùng `fnmatchcase(t, p)` — hàm có sẵn của thư viện chuẩn hiểu `*` là "bất kỳ chuỗi nào", khớp **toàn bộ** `t`. Đây là cách **độc lập** với việc ta tự tách `A`, `B`.

Ta sinh dữ liệu đúng ràng buộc: `s` dài `1..50`, bảng chữ nhỏ (`ab`, `abc`, `abcd`) để có nhiều va chạm; `p` có **đúng một** `*`. Một nửa số test lấy `p` từ **chính một chuỗi con của `s`** (đục một đoạn giữa thành `*`) để có nhiều đáp án `true`.

In [22]:
def reference(s, p):
    n = len(s)
    return any(fnmatchcase(s[i:j], p) for i in range(n + 1) for j in range(i, n + 1))


def random_case(rng):
    alphabet = rng.choice(["ab", "abc", "abcd", "abcdefghij"])
    n = rng.randint(1, 50)
    s = "".join(rng.choice(alphabet) for _ in range(n))
    if rng.random() < 0.5:                              # p lấy từ một chuỗi con của s
        i = rng.randint(0, n - 1)
        j = rng.randint(i + 1, min(n, i + 49))              # |t| <= 49 để |p| <= 50
        t = s[i:j]
        a = rng.randint(0, len(t))
        b = rng.randint(a, len(t))
        p = t[:a] + "*" + t[b:]                         # đục đoạn t[a:b] thành '*'
    else:
        m = rng.randint(0, 49)                          # p có m chữ cái + 1 dấu '*'
        letters = [rng.choice(alphabet) for _ in range(m)]
        k = rng.randint(0, m)
        p = "".join(letters[:k]) + "*" + "".join(letters[k:])
    return s, p


rng = random.Random(3407)
true_count = 0
for trial in range(5000):
    s, p = random_case(rng)
    assert 1 <= len(s) <= 50 and 1 <= len(p) <= 50 and p.count("*") == 1 and s.isalpha() and s.islower()
    ref = reference(s, p)
    true_count += ref
    for name, fn in impls.items():
        assert fn(s, p) == ref, (name, s, p)
print("PASS 5000 cặp ngẫu nhiên; số đáp án true:", true_count, "| false:", 5000 - true_count)

PASS 5000 cặp ngẫu nhiên; số đáp án true: 2730 | false: 2270


## 13. Các lỗi người mới hay gặp

### Lỗi 1: Chỉ kiểm tra `A in s and B in s`

In [23]:
def sai_chi_kiem_co_mat(s, p):
    A, B = p.split("*")
    return A in s and B in s


print("sai:", sai_chi_kiem_co_mat("ba", "a*b"), "| đúng:", Solution().hasMatch("ba", "a*b"))
print("sai:", sai_chi_kiem_co_mat("abc", "abc*bc"), "| đúng:", Solution().hasMatch("abc", "abc*bc"))

sai: True | đúng: False
sai: True | đúng: False


**Vì sao sai:** đề yêu cầu `A` đứng **trước** `B` trong cùng một chuỗi con, và hai phần **không dùng chung ký tự**. `"ba"`: có `a` và có `b` nhưng `b` đứng trước `a`. `"abc"` với `"abc*bc"`: có `abc` và có `bc` nhưng `bc` nằm **chồng** lên `abc`. **Cách sửa:** tìm `B` **từ chỉ số `i + len(A)` trở đi**.

### Lỗi 2: Tìm `B` từ đầu chuỗi (bỏ `start`)

In [24]:
def sai_find_khong_start(s, p):
    A, B = p.split("*")
    i = s.find(A)
    if i == -1:
        return False
    j = s.find(B)                              # SAI: lấy B đầu tiên của cả chuỗi
    return j != -1 and i + len(A) <= j


print("sai:", sai_find_khong_start("bab", "a*b"), "| đúng:", Solution().hasMatch("bab", "a*b"))

sai: False | đúng: True


**Vì sao sai:** ở `"bab"`, `b` **đầu tiên** nằm ở chỉ số `0` (trước `a`), nhưng còn một `b` thứ hai ở chỉ số `2` dùng được. Dùng `find(B)` (không `start`) chỉ nhìn thấy `b` đầu tiên rồi kết luận sai. **Cách sửa:** `s.find(B, i + len(A))`, hoặc dùng `s.rfind(B)` (lần xuất hiện **cuối**).

### Lỗi 3: Quên chuỗi rỗng ở hai đầu (`"*"`, `"u*"`, `"*c"`)

In [25]:
def sai_bo_qua_rong(s, p):
    A, B = p.split("*")
    if A == "" or B == "":                     # SAI: "giả sử" rỗng nghĩa là không khớp
        return False
    i = s.find(A)
    return i != -1 and s.find(B, i + len(A)) != -1


for s, p in [("luck", "u*"), ("abc", "*c"), ("abc", "*")]:
    print(f"s={s!r}, p={p!r} | sai: {sai_bo_qua_rong(s, p)} | đúng: {Solution().hasMatch(s, p)}")

s='luck', p='u*' | sai: False | đúng: True
s='abc', p='*c' | sai: False | đúng: True
s='abc', p='*' | sai: False | đúng: True


**Vì sao sai:** `'*'` thay được bằng dãy **độ dài 0 trở lên**, và `A`, `B` rỗng là tình huống bình thường (ví dụ 3 của đề chính là `"u*"`). Chuỗi rỗng là chuỗi con của mọi chuỗi và `find("")` đã xử lý đúng. **Cách sửa:** không viết nhánh riêng; để `find` lo.

### Lỗi 4: Xử lý `'*'` như một ký tự thường

In [26]:
def sai_sao_la_ky_tu(s, p):
    return p in s                              # SAI: tìm nguyên văn "ee*e" trong s


print("sai:", sai_sao_la_ky_tu("leetcode", "ee*e"), "| đúng:", Solution().hasMatch("leetcode", "ee*e"))
print("sai:", sai_sao_la_ky_tu("luck", "u*"), "| đúng:", Solution().hasMatch("luck", "u*"))

sai: False | đúng: True
sai: False | đúng: True


**Vì sao sai:** `s` chỉ có chữ cái, không bao giờ chứa `'*'`, nên `p in s` luôn `False`. **Cách sửa:** tách `p` tại `'*'` rồi khớp hai phần.

### Lỗi 5: Dùng `s.index` thay cho `s.find`

In [27]:
def sai_dung_index(s, p):
    A, B = p.split("*")
    i = s.index(A)                              # SAI: ValueError khi A không có trong s
    return s.find(B, i + len(A)) != -1


try:
    sai_dung_index("car", "z*r")
except ValueError as e:
    print("ValueError:", e)
print("đúng:", Solution().hasMatch("car", "z*r"))

ValueError: substring not found
đúng: False


**Vì sao sai:** `index` ném lỗi khi không tìm thấy, còn bài cần trả `False`. **Cách sửa:** dùng `find` và kiểm tra `-1`.

## 14. Mẹo làm bài nhanh & nhận dạng pattern

- Mẫu có **một ký tự đại diện `*`** ⇒ tách mẫu thành `A` và `B`, rồi khớp **theo thứ tự**: `A` rồi `B`.
- Với **hai phần cần xuất hiện theo thứ tự, không chồng nhau**, mẫu tham lam kinh điển: lấy lần xuất hiện **sớm nhất** của phần đầu và lần **muộn nhất** của phần cuối (hoặc tìm phần cuối từ chỗ phần đầu kết thúc).
- `str.find(sub, start)` có tham số `start`: dùng nó để "tìm tiếp từ vị trí này"; `-1` nghĩa là không có.
- Chuỗi rỗng khớp mọi chỗ; `find("")` trả đúng giá trị `start` ⇒ không cần nhánh đặc biệt.
- `s.index` ném lỗi; `s.find` trả `-1` — chọn `find` khi "không có" là một kết quả hợp lệ.
- Kiểm tra bằng **đáp án chuẩn độc lập** (`fnmatchcase` trên mọi chuỗi con) trước khi tin vào tham lam.
- Bài toán tổng quát hơn (nhiều `*`, ký tự `?`) là **Wildcard Matching (44)** cần quy hoạch động hoặc tham lam hai con trỏ.

## 15. Biến thể của bài

1. **Nhiều dấu `*`:** tách bằng `split("*")` thành nhiều phần, khớp lần lượt từng phần với `find` từ vị trí sau phần trước (chính là cách tổng quát của lời giải doocs). Nếu cần khớp **toàn bộ** `s` (không chỉ chuỗi con) thì phần đầu phải ở vị trí `0` và phần cuối phải kết thúc ở cuối `s`.
2. **Trả về chuỗi con ngắn nhất/dài nhất khớp** thay vì `True/False`.
3. **Khớp toàn bộ chuỗi** (`s` khớp `p` từ đầu tới cuối) — Wildcard Matching (44), Regular Expression Matching (10).
4. **Đếm số chuỗi con khớp** với mẫu `A*B`.
5. Bài liên quan: **Find the Index of the First Occurrence in a String (28)**, **Is Subsequence (392)**, **Wildcard Matching (44)**, **Repeated Substring Pattern (459)**.

In [28]:
def has_match_multi(s, p):
    """Nhiều dấu '*': khớp lần lượt từng phần, mỗi phần tìm từ chỗ phần trước kết thúc."""
    pos = 0
    for part in p.split("*"):
        j = s.find(part, pos)
        if j == -1:
            return False
        pos = j + len(part)
    return True


def full_match_one_star(s, p):
    """s khớp TOÀN BỘ p (một dấu *): bắt đầu bằng A, kết thúc bằng B, đủ dài."""
    A, B = p.split("*")
    return s.startswith(A) and s.endswith(B) and len(s) >= len(A) + len(B)


def shortest_match(s, p):
    A, B = p.split("*")
    best = None
    n = len(s)
    for i in range(n + 1):
        if s.startswith(A, i):
            for j in range(i + len(A), n + 1):
                if s.startswith(B, j):
                    t = s[i:j + len(B)]
                    if best is None or len(t) < len(best):
                        best = t
    return best


print(has_match_multi("leetcode", "l*e*d"))            # True
print(has_match_multi("leetcode", "d*l*e"))            # False
print(full_match_one_star("eetcode", "ee*e"), full_match_one_star("leetcode", "ee*e"))   # True False
print(shortest_match("leetcode", "e*e"))               # 'ee'
assert has_match_multi("leetcode", "l*e*d") and not has_match_multi("leetcode", "d*l*e")
assert full_match_one_star("eetcode", "ee*e") and not full_match_one_star("leetcode", "ee*e")
assert shortest_match("leetcode", "e*e") == "ee"
assert shortest_match("car", "c*v") is None

True
False
True False
ee


## 16. So sánh các cách làm

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Mọi chuỗi con | `startswith(A)` + `endswith(B)` + đủ dài | `O(n²(n+m))` | `O(n)` | bám sát định nghĩa, đáp án chuẩn |
| 2. Mọi vị trí `i`, `j` | `startswith(A, i)`, `startswith(B, j)` | `O(n²m)` | `O(1)` | rõ ràng, không dùng `find` |
| 3. Tham lam `find` hai lần | `find(A)`, rồi `find(B, i + len(A))` | `O(nm)` | `O(1)` | **nên nộp**, mở rộng được nhiều `*` |
| 4. `find` + `rfind` | `i + len(A) <= j` | `O(nm)` | `O(1)` | ngắn gọn nhất |
| 5. Regex `A.*B` | `re.search` | `O(n²)` | `O(m)` | đối chiếu, mẫu phức tạp |

## 17. 5 bài tập vận dụng (dễ → khó)

**Bài 1 (đoán output).** Không chạy code, đoán: `"a*bc".split("*")`, `"hello".find("l", 3)`, `"hello".rfind("l")`, `Solution().hasMatch("abcd", "b*d")`, `Solution().hasMatch("abcd", "d*b")`.

**Bài 2 (tính tay).** Với `s = "abcabc"`, `p = "bc*ab"`: xác định `A`, `B`, vị trí `find(A)`, nơi `A` kết thúc, rồi kết luận `true`/`false`; thử thêm `p = "bc*bc"`.

**Bài 3 (điền TODO).** Hoàn thành `has_match_todo(s, p)` theo Cách 3.

In [29]:
def has_match_todo(s, p):
    # TODO 1: tách p thành A và B bằng p.split("*")
    # TODO 2: i = s.find(A); nếu i == -1 thì return False
    # TODO 3: return s.find(B, i + len(A)) != -1
    pass

# KHÔNG gọi hàm TODO ở đây để khi Run All không bị lỗi (TODO chưa điền nên kết quả chưa đúng).

**Bài 4 (tự viết lại).** Viết `has_match_find_rfind2(s, p)` theo Cách 4 **không dùng `split`**: dùng `k = p.index("*")`, `A = p[:k]`, `B = p[k + 1:]`.

**Bài 5 (biến thể / mini challenge).** (a) Viết `count_matching_substrings(s, p)` đếm số cặp `(i, j)` với `0 <= i <= j <= n` mà `s[i:j]` khớp `p` (một dấu `*`, khớp toàn bộ `s[i:j]`). (b) Viết `has_match_two_stars(s, p)` cho mẫu có **đúng hai** dấu `*` (khớp một chuỗi con).

## 18. Gợi ý 3 mức cho bài tập

- **Mức 1 (nhẹ):** Bài 1: `['a', 'bc']`, `3`, `3`, `True` (`A = "b"` tại `1`, `B = "d"` tại `3`), `False` (`d` ở cuối, không có `b` phía sau). Bài 3: `A, B = p.split("*")` rồi làm theo mục 9.
- **Mức 2 (vừa):** Bài 2: `A = "bc"`, `B = "ab"`; `find(A) = 1`, `A` kết thúc tại `3`; `find("ab", 3) = 3` ⇒ `true` (chuỗi con `"bcab"`). Với `p = "bc*bc"`: `B = "bc"`, `find("bc", 3) = 4` ⇒ `true`. Bài 4: `k = p.index("*")`.
- **Mức 3 (sâu):** Bài 5(a): hai vòng `for i, j` và `fnmatchcase(s[i:j], p)` hoặc điều kiện `startswith/endswith/len`. 5(b): `parts = p.split("*")` có `3` phần; khớp lần lượt bằng `pos`, giống `has_match_multi` ở mục 15.

## 19. Đáp án tham khảo cho bài tập

Chạy kiểm tra bằng `assert`.

In [30]:
# Bài 1
assert "a*bc".split("*") == ["a", "bc"]
assert "hello".find("l", 3) == 3
assert "hello".rfind("l") == 3
assert Solution().hasMatch("abcd", "b*d") is True
assert Solution().hasMatch("abcd", "d*b") is False

# Bài 2
A, B = "bc*ab".split("*")
assert (A, B) == ("bc", "ab")
assert "abcabc".find(A) == 1 and "abcabc".find(B, 1 + len(A)) == 3
assert Solution().hasMatch("abcabc", "bc*ab") is True
assert Solution().hasMatch("abcabc", "bc*bc") is True


# Bài 3
def has_match_todo_ans(s, p):
    A, B = p.split("*")
    i = s.find(A)
    if i == -1:
        return False
    return s.find(B, i + len(A)) != -1


for s, p, expected in examples:
    assert has_match_todo_ans(s, p) == expected


# Bài 4
def has_match_find_rfind2(s, p):
    k = p.index("*")
    A, B = p[:k], p[k + 1:]
    i, j = s.find(A), s.rfind(B)
    return i != -1 and j != -1 and i + len(A) <= j


for s, p, expected in examples:
    assert has_match_find_rfind2(s, p) == expected
assert has_match_find_rfind2("bab", "a*b") is True and has_match_find_rfind2("ba", "a*b") is False


# Bài 5
def count_matching_substrings(s, p):
    n = len(s)
    return sum(fnmatchcase(s[i:j], p) for i in range(n + 1) for j in range(i, n + 1))


def has_match_two_stars(s, p):
    A, B, C2 = p.split("*")
    pos = s.find(A)
    if pos == -1:
        return False
    pos += len(A)
    mid = s.find(B, pos)
    if mid == -1:
        return False
    return s.find(C2, mid + len(B)) != -1


assert count_matching_substrings("abab", "a*b") == 3        # "ab", "ab", "abab"
assert count_matching_substrings("luck", "u*") == 3          # "u", "uc", "uck"
assert has_match_two_stars("leetcode", "l*e*d") is True
assert has_match_two_stars("leetcode", "d*l*e") is False
print("PASS đáp án bài tập")

PASS đáp án bài tập


## 20. Template nhớ nhanh + Checklist tự kiểm tra trước khi submit

**Template** (mẫu một dấu `*`: tách thành `A`, `B`; `A` sớm nhất, `B` từ chỗ `A` kết thúc)

```python
class Solution:
    def hasMatch(self, s: str, p: str) -> bool:
        A, B = p.split("*")
        i = s.find(A)
        if i == -1:
            return False
        return s.find(B, i + len(A)) != -1
```

**Checklist**

- [ ] Đã tách `p` thành `A` (trước `*`) và `B` (sau `*`)?
- [ ] `A` xuất hiện **trước** `B` và hai phần **không chồng nhau** (`find(B, i + len(A))`)?
- [ ] Dùng `find` (trả `-1`) chứ không dùng `index` (ném lỗi)?
- [ ] Đã thử `A` hoặc `B` rỗng (`"u*"`, `"*c"`) và `p = "*"` (luôn `True`)?
- [ ] Đã thử phản ví dụ `("ba", "a*b")` (`False`), `("bab", "a*b")` (`True`), `("abc", "abc*bc")` (`False`)?
- [ ] Skeleton đúng `class Solution` / `def hasMatch(self, s: str, p: str) -> bool`?